# nb_silver_to_gold
**Phase 1C** - build business-ready Gold tables with Spark SQL, then tune the Delta tables.

Gold tables:
| Table | Grain | Used by |
|---|---|---|
| `gold_sales_by_day_region` | day x region x category | Power BI, Direct Lake |
| `gold_top_customers` | top 20 customers by lifetime revenue | Power BI |
| `gold_sales_by_store_month` | store x month x channel | Power BI |
| `gold_dim_date` | one row per calendar day | Warehouse `dim_date` |

In [1]:
run_date = "2026-06-13"

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

# Check the dimension schemas so the joins below use real column names
for t in ["silver_dim_store", "silver_dim_product", "silver_dim_customer", "silver_sales"]:
    print(t); spark.table(t).printSchema()

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 4, Finished, Available, Finished, False)

silver_dim_store
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- open_date: date (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

silver_dim_product
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

silver_dim_customer
root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- loyalty_tier: string (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

silver_sales
root
 |-- order_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- customer_id: integer (nullable = true

In [3]:
spark.sql("""
CREATE OR REPLACE TABLE gold_sales_by_day_region AS
SELECT s.order_date,
       st.region,
       p.category,
       SUM(s.net_amount) AS revenue,
       SUM(s.quantity)   AS units,
       COUNT(*)          AS orders
FROM silver_sales s
JOIN silver_dim_store   st ON s.store_id   = st.store_id
JOIN silver_dim_product p  ON s.product_id = p.product_id
GROUP BY s.order_date, st.region, p.category
""")
print("gold_sales_by_day_region:", spark.table("gold_sales_by_day_region").count(), "rows")

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 5, Finished, Available, Finished, False)

gold_sales_by_day_region: 1947 rows


In [4]:
# top 20 customers by lifetime revenue, with name + tier
spark.sql("""
CREATE OR REPLACE TABLE gold_top_customers AS
SELECT c.customer_id, c.customer_name, c.loyalty_tier,
       SUM(s.net_amount) AS lifetime_revenue,
       COUNT(*)          AS orders
FROM silver_sales s
JOIN silver_dim_customer c ON s.customer_id = c.customer_id
GROUP BY c.customer_id, c.customer_name, c.loyalty_tier
ORDER BY lifetime_revenue DESC
LIMIT 20
""")
display(spark.table("gold_top_customers"))

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f9835b53-1308-4951-9470-0b7cc3969588)

In [5]:
# Extra Gold tables (store x month x channel) and a date dimension for the warehouse
spark.sql("""
CREATE OR REPLACE TABLE gold_sales_by_store_month AS
SELECT st.store_id, st.store_name, st.region,
       date_trunc('month', s.order_date) AS month_start,
       s.channel,
       SUM(s.net_amount) AS revenue, SUM(s.quantity) AS units, COUNT(*) AS orders
FROM silver_sales s
JOIN silver_dim_store st ON s.store_id = st.store_id
GROUP BY st.store_id, st.store_name, st.region, date_trunc('month', s.order_date), s.channel
""")
print("Gold sales by store and month:",
      spark.table("gold_sales_by_store_month").count(), "rows")

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 7, Finished, Available, Finished, False)

Gold sales by store and month: 90 rows


In [6]:
# take the distinct order dates available in silver_sales and create a date dimension from them
spark.sql("""
CREATE OR REPLACE TABLE gold_dim_date AS
SELECT DISTINCT
    CAST(DATE_FORMAT(order_date, 'yyyyMMdd') AS INT) AS date_key,
    order_date AS full_date,
    YEAR(order_date) AS year,
    QUARTER(order_date) AS quarter,
    MONTH(order_date) AS month,
    DAY(order_date) AS day,
    DATE_FORMAT(order_date, 'EEEE') AS day_name
FROM silver_sales
WHERE order_date IS NOT NULL
""")

print("Gold date dimension:",
      spark.table("gold_dim_date").count(), "rows")
display(spark.table("gold_dim_date"))

StatementMeta(, ec6dd6e6-9344-4eae-8cbf-0e74a0ec04d6, 8, Finished, Available, Finished, False)

Gold date dimension: 152 rows


SynapseWidget(Synapse.DataFrame, 77f6662b-e0fb-450e-91e9-a862dd161c99)

### Delta performance tuning
- **Small-file problem** - many small writes leave many tiny Parquet files; queries open all of them. `OPTIMIZE` compacts them.
- **V-Order** - an optimisation that makes Parquet faster for Power BI Direct Lake and the SQL endpoint.
- **Z-order** - Organizes data within Delta Lake files so that rows with similar values (e.g., order_date) are stored closer together
- **VACUUM** - deletes data files no longer referenced by the log. The default **7-day (168 h) retention** protects time travel and concurrent readers

In [ ]:
spark.sql("OPTIMIZE gold_sales_by_day_region VORDER")
spark.sql("OPTIMIZE silver_sales ZORDER BY (order_date)")
spark.sql("OPTIMIZE gold_sales_by_store_month VORDER")
print("OPTIMIZE completed without error")
display(spark.sql("DESCRIBE DETAIL silver_sales"))     

In [ ]:
# Retention-safe cleanup. Keeps 7 days so time travel still works.
spark.sql("VACUUM silver_sales RETAIN 168 HOURS")
spark.sql("VACUUM gold_sales_by_day_region RETAIN 168 HOURS")

In [ ]:
# Reconcile Gold to Silver: any difference means sales rows had no matching store/product (orphans)
s = spark.sql("SELECT ROUND(SUM(net_amount),2) AS v FROM silver_sales").first()["v"]
g = spark.sql("SELECT ROUND(SUM(revenue),2) AS v FROM gold_sales_by_day_region").first()["v"]
print(f"Silver revenue={s} | Gold revenue={g} | difference={round(s - g, 2)}")

log = spark.createDataFrame(
    [(run_date, "silver_to_gold", "silver_revenue", float(s)),
     (run_date, "silver_to_gold", "gold_revenue", float(g)),
     (run_date, "silver_to_gold", "revenue_diff", float(s - g))],
    "run_date string, step string, metric string, value double"
).withColumn("run_ts", F.current_timestamp())
log.write.mode("append").format("delta").saveAsTable("ops_dq_log")

```sql
SELECT TOP 10 * FROM gold_sales_by_day_region ORDER BY revenue DESC;
SELECT * FROM gold_top_customers;
```